In [23]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.svm import SVR, SVC
from lightgbm import LGBMClassifier
from scikeras.wrappers import KerasRegressor   ### KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [24]:
def create_sequences_by_participant(df, window_size, feature_cols, target_col):
    X_list, y_list, groups_list = [], [], []
    
    for p_id, group in df.groupby("participant"):
        # Se il paziente ha abbastanza dati per almeno una finestra
        if len(group) >= window_size:
            features = group[feature_cols].values
            target = group[target_col].values
            
            # Crea sequenze solo per questo paziente
            for i in range(len(group) - window_size):
                X_list.append(features[i : i + window_size])
                y_list.append(target[i + window_size])
                groups_list.append(p_id) # Salva il partecipante corretto
                
    return np.array(X_list), np.array(y_list), np.array(groups_list)

In [25]:
def build_lstm_model(n_features, timesteps, n_outputs=3, n_units=16, dropout=0.2, lr=1e-3):
    """Regressione multi-uscita: SIS, OHS, OKS (n_outputs=3)."""
    model = Sequential([
        LSTM(n_units, input_shape=(timesteps, n_features)),  ## n_units: 32, 64
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(n_outputs),
    ])

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"],
    )

    return model

In [26]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [27]:
df = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [28]:
ana = pd.read_csv('./new_dataset/maison-llf-demographics.CSV', sep=",")

In [29]:
ana_col = list(ana.columns)

In [30]:
ana_encoded = ana[["participant", "age", "sex", "education", "work", "fracture-type", "ethnicity"]].copy()

# male=1, female=0
ana_encoded["sex_male"] = ana_encoded["sex"].map({"male": 1, "female": 0})

# Education label encoding with doctorate as highest level
education_order = {
    "secondary education": 0,
    "undergraduate degree": 1,
    "graduate degree": 2,
    "doctorate degree": 3
}
ana_encoded["education_label"] = ana_encoded["education"].map(education_order)

# retired=0, employed part-time=1
ana_encoded["work_part_time"] = ana_encoded["work"].map({"retired": 0, "employed part-time": 1})

fracture_dummies = pd.get_dummies(
    ana_encoded["fracture-type"],
    prefix="fracture",
    dtype=int
)

ethnicity_dummies = pd.get_dummies(
    ana_encoded["ethnicity"],
    prefix="ethnicity",
    dtype=int
)

num_ana = pd.concat(
    [
        ana_encoded[["participant", "age", "sex_male", "education_label", "work_part_time"]],
        fracture_dummies,
        ethnicity_dummies,
    ],
    axis=1,
)

data = df.merge(num_ana, how='right', on='participant')

In [31]:
data.head(5)

,participant,timestamp,clinical-timestamp,sis-01,sis-02,sis-03,sis-04,sis-05,sis-06,sis,...,sex_male,education_label,work_part_time,fracture_femur fracture,fracture_hip fracture,fracture_hip replacement,fracture_pelvis fracture,ethnicity_black,ethnicity_caucasian,ethnicity_south asian
0,1,2022-03-31,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
1,1,2022-04-01,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
2,1,2022-04-02,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
3,1,2022-04-03,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0
4,1,2022-04-04,2022-04-13,5,5,4,4,5,2,25,...,0,1,0,0,1,0,0,1,0,0


In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 95 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   participant                            1008 non-null   int64  
 1   timestamp                              1008 non-null   object 
 2   clinical-timestamp                     1008 non-null   object 
 3   sis-01                                 1008 non-null   int64  
 4   sis-02                                 1008 non-null   int64  
 5   sis-03                                 1008 non-null   int64  
 6   sis-04                                 1008 non-null   int64  
 7   sis-05                                 1008 non-null   int64  
 8   sis-06                                 1008 non-null   int64  
 9   sis                                    1008 non-null   int64  
 10  ohs-01                                 1008 non-null   int64  
 11  ohs-

In [36]:
data[['age', 'sis', 'oks']].describe()

,age,sis,oks
count,1008.000000,1008.000000,1008.000000
mean,76.500000,23.861111,30.222222
std,8.646885,3.836344,9.689019
min,60.000000,15.000000,15.000000
25%,71.000000,21.750000,22.750000
50%,75.500000,24.000000,28.500000
75%,85.000000,27.000000,35.750000
max,94.000000,30.000000,48.000000


In [12]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(data["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(data["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(data["oks"], [25, 50, 75])

In [13]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [14]:
# Apply discretization
data["SISS_Category_Q"] = pd.cut(data["sis"], bins=[data["sis"].min(), siss_q1, siss_q2, siss_q3, data["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
data["OHSS_Category_Q"] = pd.cut(data["ohs"], bins=[data["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, data["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

data["OKSS_Category_Q"] = pd.cut(data["oks"], bins=[data["oks"].min(), okss_q1, okss_q2, okss_q3, data["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [15]:
# Extract only numeric features for LOPO (drop timestamps/string columns).
exclude_cols = [
    "participant",
    "timestamp",
    "clinical-timestamp",
    "motion-max-timestamp",
    "step-max-timestamp",
    "SISS_Category_Q",
    "OHSS_Category_Q",
    "OKSS_Category_Q",
]

feature_cols = [c for c in data.columns if c not in exclude_cols]
X = data[feature_cols].select_dtypes(include=[np.number]).copy()
groups = data["participant"]

# Conta i record per ogni partecipante
counts = df.groupby("participant").size()

WINDOW_SIZE = 56

In [16]:
nct = df.groupby('participant')['clinical-timestamp'].nunique()
print()
print('Visite distinte (clinical-timestamp) per partecipante:', nct.to_string())


Visite distinte (clinical-timestamp) per partecipante: participant
1     4
2     4
3     4
4     4
5     4
6     4
7     4
8     4
9     4
10    4
11    4
12    4
13    4
14    4
15    4
16    4
17    4
18    4


In [17]:
counts

participant
1     56
2     56
3     56
4     56
5     56
6     56
7     56
8     56
9     56
10    56
11    56
12    56
13    56
14    56
15    56
16    56
17    56
18    56
dtype: int64

In [18]:
# Define classifier and hyperparameter grid
param_grid = {
            "model__n_units": [16, 32, 64],
            "model__dropout": [0.0, 0.2],
            "model__lr": [1e-3, 3e-4],
            "batch_size": [16, 32],
            "epochs": [10, 20],
        }

In [19]:
# Leave-One-Patient-Out CV (LOPO)
outer_logo = LeaveOneGroupOut()

In [20]:
# Metriche per fold (regressione su SIS, OHS, OKS)
SCORE_TARGETS = ["SIS", "OHS", "OKS"]
performance_metrics = []

In [21]:
# ESEMPIO D'USO:
WINDOW_SIZE = 7 # 4-14-28-56

lista_features = X.columns

X_input, y_input, groups_input = create_sequences_by_participant(
    data, WINDOW_SIZE, lista_features, ['sis', 'ohs', 'oks']
)

# Ora le lunghezze saranno TUTTE uguali e coerenti
print(X_input.shape[0], y_input.shape[0], groups_input.shape[0])

882 882 882


In [22]:
data.shape

(1008, 98)

In [22]:
X_input.shape 

(882, 7, 90)

In [23]:
len(X_input[0][0])

90

In [24]:
y_input  ### 18 pazienti e 3 var risposta

array([[25, 25, 31],
       [25, 25, 31],
       [25, 25, 31],
       ...,
       [30, 23, 32],
       [30, 23, 32],
       [30, 23, 32]], shape=(882, 3))

In [24]:
# Outer LOPO Loop
count=0
y_input = y_input 

for train_idx, test_idx in outer_logo.split(X_input, y_input, groups_input):
    #print(train_idx) index
    count=count+1
    print(count)
    
    #if count == 6 or count == 7:
    #    continue 
    X_train_outer, X_test = X_input[train_idx], X_input[test_idx] #X_train_outer, X_test = X_input.loc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y_input[train_idx], y_input[test_idx] #y_train_outer, y_test = y_input.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups_input[train_idx] #groups_train_outer = groups.iloc[train_idx]
    print(np.unique(groups_train_outer)) #print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer[inner_train_idx] #groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(np.unique(groups_train_inner)) #print(groups_train_inner.unique())
        
        # Hyperparameter tuning (massimizza R² medio sul validation set)
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = KerasRegressor(
                model=build_lstm_model,
                n_features=len(lista_features),
                timesteps=WINDOW_SIZE,
                n_outputs=y_input.shape[1],
                epochs=10,
                batch_size=32,
                verbose=0,
            )
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = r2_score(y_val, y_val_pred, multioutput="uniform_average")

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params

    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    mae_each = mean_absolute_error(y_test, y_pred, multioutput="raw_values")
    rmse_avg = np.sqrt(mean_squared_error(y_test, y_pred, multioutput="uniform_average"))
    r2_avg = r2_score(y_test, y_pred, multioutput="uniform_average")

    performance_metrics.append(np.concatenate([mae_each, [rmse_avg, r2_avg]]))


1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  7  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  8  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  9 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8 10 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 16 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 17 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 18]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
2
[ 1  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 

In [ ]:
metric_cols = [f"MAE_{n}" for n in SCORE_TARGETS] + ["RMSE_avg", "R2_avg"]
performance_df = pd.DataFrame(performance_metrics, columns=metric_cols)
performance_df

In [ ]:
performance_df.describe()

In [ ]:
performance_df

In [ ]:
assert False

In [ ]:
# Salva metriche di regressione (SIS, OHS, OKS)
output_path = os.path.join(root, "new_results/model_performance_scores.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [ ]:
# MAE medio per score su tutti i fold
avg_mae = performance_df[[f"MAE_{n}" for n in SCORE_TARGETS]].mean()
avg_mae.index = pd.Index(SCORE_TARGETS, name="score")
plt.figure(figsize=(6, 4))
avg_mae.plot(kind="bar", color=["#4c72b0", "#55a868", "#c44e52"])
plt.ylabel("MAE medio")
plt.title("Errore assoluto medio per score (LOPO)")
plt.tight_layout()
plot_path = os.path.join(root, "new_results/lstm_mae_by_score.pdf")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"\n✅ Metriche salvate in: {output_path}")
print(f"✅ Grafico MAE salvato in: {plot_path}")